# Tesla and GameStop Stock & Revenue Analysis

This project analyzes Tesla and GameStop stock price and revenue data using Python. Stock data was collected with `yfinance`, while revenue data was extracted using web scraping.

## 1. Import Libraries

In [22]:
import yfinance as yf
import requests
import pandas as pd
from bs4 import BeautifulSoup as bs

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

## 2. Tesla Data Collection and Cleaning

In [23]:
tesla = yf.Ticker("TSLA")

tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)

tesla_stock = tesla_data[["Date", "Close"]].copy()
tesla_stock.rename(columns={"Close": "Stock Price"}, inplace=True)

tesla_stock["Date"] = pd.to_datetime(tesla_stock["Date"]).dt.date

tesla_stock.head()

,Date,Stock Price
0,2010-06-29,1.592667
1,2010-06-30,1.588667
2,2010-07-01,1.464000
3,2010-07-02,1.280000
4,2010-07-06,1.074000


In [24]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"

html_data = requests.get(url).text
soup = bs(html_data, "html.parser")

tesla_revenue = pd.DataFrame(columns=["Date", "Revenue"])

for row in soup.find_all("tbody")[1].find_all("tr"):
    col = row.find_all("td")
    
    if len(col) > 0:
        date = col[0].text.strip()
        revenue = col[1].text.strip()
        tesla_revenue.loc[len(tesla_revenue)] = [date, revenue]

tesla_revenue["Revenue"] = tesla_revenue["Revenue"].str.replace("$", "", regex=False)
tesla_revenue["Revenue"] = tesla_revenue["Revenue"].str.replace(",", "", regex=False)
tesla_revenue["Revenue"] = tesla_revenue["Revenue"].str.strip()

tesla_revenue = tesla_revenue[tesla_revenue["Revenue"] != ""].copy()

tesla_revenue["Revenue"] = tesla_revenue["Revenue"].astype("int")
tesla_revenue["Date"] = pd.to_datetime(tesla_revenue["Date"])

tesla_revenue.head()

,Date,Revenue
0,2022-09-30,21454
1,2022-06-30,16934
2,2022-03-31,18756
3,2021-12-31,17719
4,2021-09-30,13757


## 3. GameStop Data Collection and Cleaning

In [25]:
gme = yf.Ticker("GME")

gme_data = gme.history(period="max")
gme_data.reset_index(inplace=True)

gme_stock = gme_data[["Date", "Close"]].copy()
gme_stock.rename(columns={"Close": "Stock Price"}, inplace=True)

gme_stock["Date"] = pd.to_datetime(gme_stock["Date"]).dt.date

gme_stock.head()

,Date,Stock Price
0,2002-02-13,1.691666
1,2002-02-14,1.683250
2,2002-02-15,1.674834
3,2002-02-19,1.607504
4,2002-02-20,1.662210


In [26]:
url = "https://www.macrotrends.net/stocks/charts/GME/gamestop/revenue"

headers = {"User-Agent": "Mozilla/5.0"}

html_data = requests.get(url, headers=headers).text
soup = bs(html_data, "html.parser")

gme_revenue = pd.DataFrame(columns=["Date", "Revenue"])

for row in soup.find_all("tbody")[1].find_all("tr"):
    col = row.find_all("td")
    
    if len(col) > 0:
        date = col[0].text.strip()
        revenue = col[1].text.strip()
        gme_revenue.loc[len(gme_revenue)] = [date, revenue]

gme_revenue["Revenue"] = gme_revenue["Revenue"].str.replace("$", "", regex=False)
gme_revenue["Revenue"] = gme_revenue["Revenue"].str.replace(",", "", regex=False)
gme_revenue["Revenue"] = gme_revenue["Revenue"].str.strip()

gme_revenue = gme_revenue[gme_revenue["Revenue"] != ""].copy()

gme_revenue["Revenue"] = gme_revenue["Revenue"].astype("int")
gme_revenue["Date"] = pd.to_datetime(gme_revenue["Date"])

gme_revenue.head()

,Date,Revenue
0,2026-01-31,1104
1,2025-10-31,821
2,2025-07-31,972
3,2025-04-30,732
4,2025-01-31,1283


In [27]:
start_date = pd.to_datetime("2020-01-01").date()

tesla_stock_filtered = tesla_stock[tesla_stock["Date"] >= start_date].copy()
gme_stock_filtered = gme_stock[gme_stock["Date"] >= start_date].copy()

tesla_revenue_filtered = tesla_revenue[tesla_revenue["Date"].dt.date >= start_date].copy()
gme_revenue_filtered = gme_revenue[gme_revenue["Date"].dt.date >= start_date].copy()

## 4. Data Preparation
The stock data was converted to quarterly average prices and merged with quarterly revenue data.

In [28]:
tesla_stock_filtered["Quarter"] = pd.to_datetime(tesla_stock_filtered["Date"]).dt.to_period("Q")
gme_stock_filtered["Quarter"] = pd.to_datetime(gme_stock_filtered["Date"]).dt.to_period("Q")

tesla_revenue_filtered["Quarter"] = pd.to_datetime(tesla_revenue_filtered["Date"]).dt.to_period("Q")
gme_revenue_filtered["Quarter"] = pd.to_datetime(gme_revenue_filtered["Date"]).dt.to_period("Q")

tesla_quarterly = tesla_stock_filtered.groupby("Quarter")["Stock Price"].mean().reset_index()
gme_quarterly = gme_stock_filtered.groupby("Quarter")["Stock Price"].mean().reset_index()

tesla_final = pd.merge(
    tesla_quarterly,
    tesla_revenue_filtered[["Quarter", "Revenue"]],
    on="Quarter"
)

gme_final = pd.merge(
    gme_quarterly,
    gme_revenue_filtered[["Quarter", "Revenue"]],
    on="Quarter"
)

gme_final = gme_final[gme_final["Quarter"] <= "2022Q3"].copy()

tesla_final.reset_index(drop=True, inplace=True)
gme_final.reset_index(drop=True, inplace=True)

print(tesla_final)
print(gme_final)

   Quarter  Stock Price  Revenue
0   2020Q1    41.455441     5985
1   2020Q2    54.097365     6036
2   2020Q3   118.069135     8771
3   2020Q4   170.650364    10744
4   2021Q1   251.061967    10389
5   2021Q2   217.086086    11958
6   2021Q3   235.365522    13757
7   2021Q4   335.389687    17719
8   2022Q1   311.467256    18756
9   2022Q2   272.959410    16934
10  2022Q3   279.270782    21454
   Quarter  Stock Price  Revenue
0   2020Q1     1.070121     2194
1   2020Q2     1.153135     1021
2   2020Q3     1.445039      942
3   2020Q4     3.440937     1005
4   2021Q1    29.521394     2122
5   2021Q2    48.508174     1277
6   2021Q3    45.698359     1183
7   2021Q4    45.468906     1297
8   2022Q1    29.565202     2254
9   2022Q2    31.806008     1378
10  2022Q3    32.640117     1136


## 5. Stock Price and Revenue Visualization

In [29]:
def make_graph(final_data, stock_name):
    fig = make_subplots(
        rows=2,
        cols=1,
        subplot_titles=(
            f"{stock_name} Quarterly Average Stock Price",
            f"{stock_name} Quarterly Revenue"
        ),
        shared_xaxes=True,
        vertical_spacing=0.18
    )

    fig.add_trace(
        go.Scatter(
            x=final_data["Quarter"].astype(str),
            y=final_data["Stock Price"],
            mode="lines+markers",
            name="Stock Price"
        ),
        row=1,
        col=1
    )

    fig.add_trace(
        go.Bar(
            x=final_data["Quarter"].astype(str),
            y=final_data["Revenue"],
            name="Revenue"
        ),
        row=2,
        col=1
    )

    fig.update_layout(
        title=f"{stock_name}: Stock Price vs Revenue Analysis",
        height=750,
        template="plotly_white",
        showlegend=True,
        hovermode="x unified"
    )

    fig.update_yaxes(title_text="Average Stock Price ($)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue", row=2, col=1)
    fig.update_xaxes(title_text="Quarter", row=2, col=1)

    fig.show()


make_graph(tesla_final, "Tesla")
make_graph(gme_final, "GameStop")

In [30]:
tesla_corr = tesla_final["Stock Price"].corr(tesla_final["Revenue"])
gme_corr = gme_final["Stock Price"].corr(gme_final["Revenue"])

print("Tesla Correlation:", round(tesla_corr, 3))
print("GameStop Correlation:", round(gme_corr, 3))

Tesla Correlation: 0.895
GameStop Correlation: 0.052


## 6. Correlation Analysis
This section examines whether quarterly revenue and average quarterly stock price move together.

In [31]:
def correlation_graph(final_data, stock_name):
    corr = final_data["Stock Price"].corr(final_data["Revenue"])

    fig = px.scatter(
        final_data,
        x="Revenue",
        y="Stock Price",
        text=final_data["Quarter"].astype(str),
        trendline="ols",
        title=f"{stock_name}: Revenue vs Stock Price Relationship<br><sup>Correlation: {corr:.3f}</sup>",
        labels={
            "Revenue": "Quarterly Revenue",
            "Stock Price": "Average Quarterly Stock Price ($)"
        }
    )

    fig.update_traces(
        marker=dict(size=11, opacity=0.8),
        textposition="top center"
    )

    fig.update_layout(
        template="plotly_white",
        height=650,
        title_x=0.5,
        hovermode="closest",
        showlegend=False
    )

    fig.update_xaxes(showgrid=True)
    fig.update_yaxes(showgrid=True)

    fig.show()


correlation_graph(tesla_final, "Tesla")
correlation_graph(gme_final, "GameStop")

## Conclusion

Tesla showed a strong positive relationship between quarterly revenue and stock price.

GameStop showed almost no meaningful relationship between revenue and stock price, suggesting that its stock movement was less connected to revenue fundamentals during the analyzed period.